<div align="center">

# 🌾 agrometeorologiapy

### Fórmulas de agrometeorologia em Python — um tour guiado, função por função

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/agrometeorologiapy/blob/main/examples/tutorial_colab.ipynb)
[![PyPI](https://img.shields.io/pypi/v/agrometeorologiapy.svg)](https://pypi.org/project/agrometeorologiapy/)
[![License: BSD-3-Clause](https://img.shields.io/badge/license-BSD--3--Clause-blue.svg)](https://github.com/fcoliveira-utfpr/agrometeorologiapy/blob/main/LICENSE)

</div>

---

Este notebook instala o pacote **`agrometeorologiapy`** direto do PyPI e percorre
**cada uma das suas funções**, com um pequeno enunciado (situação prática) e um
exemplo numérico executável.

Todo o notebook segue um único cenário fictício — a **Fazenda Santa Rita**, em
Ponta Grossa (PR) — para que os resultados de uma seção sirvam de entrada para
a próxima, como aconteceria numa análise agrometeorológica real.

> 💡 **Como usar:** clique em *"Open in Colab"* acima, depois em
> **Ambiente de execução → Executar tudo** (ou vá rodando célula a célula com
> `Shift+Enter`). Não precisa instalar nada na sua máquina.

## 🔧 Instalação

Instalando a última versão publicada no PyPI:

In [1]:
!pip install -q agrometeorologiapy

In [2]:
import agrometeorologiapy as amp
import pandas as pd

print("agrometeorologiapy", amp.__version__, "instalado com sucesso ✅")

agrometeorologiapy 0.1.0 instalado com sucesso ✅


## 📍 Cenário de referência

**Fazenda Santa Rita**, região de Ponta Grossa – PR.

| Variável | Valor | Significado |
|---|---|---|
| `lat` | -25.09° | Latitude (negativa = hemisfério sul) |
| `alt` | 875 m | Altitude da estação |
| `dia`, `mes` | 15, 9 | Data de referência (15 de setembro) |
| `hora`, `minuto` | 14, 20 | Horário da observação |
| `altura_poste` | 4 m | Altura de um poste/objeto, para o exemplo de sombra |
| `Tmax`, `Tmin` | 27.5 °C, 14.2 °C | Temperaturas extremas do dia |
| `UR` | 58 % | Umidade relativa do ar |
| `u2` | 1.8 m/s | Velocidade do vento a 2 m de altura |

Essas variáveis são usadas ao longo de todo o notebook — rode a célula abaixo
uma vez e siga em frente.

In [3]:
# Estação de referência: Fazenda Santa Rita, Ponta Grossa - PR
lat = -25.09
alt = 875
dia, mes = 15, 9
hora, minuto = 14, 20
altura_poste = 4
Tmax, Tmin = 27.5, 14.2
UR = 58
u2 = 1.8

---
## 1️⃣ Funções auxiliares de trigonometria

As fórmulas de agrometeorologia trabalham com ângulos em **graus**, não em
radianos. `sind`, `cosd`, `tand` e `acosd` são atalhos usados internamente por
quase todas as outras funções do pacote — mas também estão disponíveis para
você usar diretamente.

> 📌 **Situação:** você quer conferir rapidamente o seno, cosseno e tangente
> de um ângulo de 37°, e o arco-cosseno de 0,6 (já em graus, sem se preocupar
> com radianos).

In [4]:
angulo = 37
print("sind(37°) =", amp.sind(angulo))
print("cosd(37°) =", amp.cosd(angulo))
print("tand(37°) =", amp.tand(angulo))
print("acosd(0.6) =", amp.acosd(0.6), "graus")

sind(37°) = 0.6018150231520483
cosd(37°) = 0.7986355100472928
tand(37°) = 0.7535540501027942
acosd(0.6) = 53.13010235415599 graus


---
## 2️⃣ Radiação Solar

Nesta seção calculamos, passo a passo, a posição do Sol e a radiação que
chega à Fazenda Santa Rita no dia 15 de setembro. Cada função usa o resultado
da anterior — é assim que normalmente se calcula radiação solar na prática.

### 2.1 `nda` — Número do Dia do Ano

> 📌 Antes de tudo, você precisa saber em que dia do ano cai 15 de setembro,
> para usar nas fórmulas de declinação solar.

In [5]:
NDA = amp.nda(dia, mes)
print(f"Número do Dia do Ano (NDA): {NDA}")

Número do Dia do Ano (NDA): 258


### 2.2 `declinacao_solar` — Declinação solar

> 📌 Com o NDA em mãos, calcule o ângulo de declinação solar do dia — ele
> mede o quanto o Sol está "deslocado" do equador celeste nessa época do ano.

In [6]:
delta = amp.declinacao_solar(NDA)
print(f"Declinação solar (delta): {delta:.4f}°")

Declinação solar (delta): 1.8147°


### 2.3 `angulo_horario` — Ângulo horário

> 📌 A observação foi feita às 14h20. Converta esse horário local em ângulo
> horário (0° ao meio-dia solar).

In [7]:
h = amp.angulo_horario(hora, minuto)
print(f"Ângulo horário (h): {h:.2f}°")

Ângulo horário (h): 35.00°


### 2.4 `angulo_zenital` — Ângulo zenital (meio-dia solar)

> 📌 Qual é a menor altura zenital do Sol nesse dia, na latitude da fazenda?

In [8]:
Z = amp.angulo_zenital(lat, delta)
print(f"Ângulo zenital (Z): {Z:.2f}°")

Ângulo zenital (Z): 26.90°


### 2.5 `azimute_solar` — Azimute solar

> 📌 Em que direção (em relação à linha Norte-Sul) o Sol está nesse instante?

In [9]:
alfa = amp.azimute_solar(lat, delta, Z)
print(f"Azimute solar (alfa): {alfa:.2f}°")

Azimute solar (alfa): 180.00°


### 2.6 `comprimento_sombra` — Comprimento da sombra

> 📌 Há um poste de 4 m de altura no pátio da fazenda. Qual o comprimento da
> sombra que ele projeta nesse horário?

In [10]:
S = amp.comprimento_sombra(altura_poste, Z)
print(f"Comprimento da sombra (S): {S:.2f} m")

Comprimento da sombra (S): 2.03 m


### 2.7 `angulo_horario_nascer` — Ângulo horário no nascer do Sol

> 📌 A que ângulo horário o Sol nasce, nessa latitude e nessa época do ano?

In [11]:
Hn = amp.angulo_horario_nascer(lat, delta)
print(f"Ângulo horário no nascer do Sol (Hn): {Hn:.2f}°")

Ângulo horário no nascer do Sol (Hn): 89.15°


### 2.8 `fotoperiodo` — Fotoperíodo (duração do dia)

> 📌 Quantas horas de luz solar esse dia tem, do nascer ao pôr do Sol?

In [12]:
N = amp.fotoperiodo(Hn)
print(f"Fotoperíodo (N): {N:.2f} horas")

Fotoperíodo (N): 11.89 horas


### 2.9 `fator_correcao_distancia` — Correção da distância Terra-Sol

> 📌 A Terra não está sempre à mesma distância do Sol. Qual a correção
> (d/D)² aplicável a este dia do ano?

In [13]:
dD2 = amp.fator_correcao_distancia(NDA)
print(f"Fator de correção (d/D)^2: {dD2:.4f}")

Fator de correção (d/D)^2: 0.9912


### 2.10 `irradiancia_extraterrestre` — Radiação no topo da atmosfera (Qo)

> 📌 Antes de qualquer atenuação pela atmosfera, quanta energia solar chega
> ao topo da atmosfera, sobre a fazenda, nesse dia?

In [14]:
Qo = amp.irradiancia_extraterrestre(lat, delta, Hn, dD2)
print(f"Irradiância solar extraterrestre diária (Qo): {Qo:.2f} MJ/m² dia")

Irradiância solar extraterrestre diária (Qo): 32.95 MJ/m² dia


### 2.11 `insolacao` — Estimativa do brilho solar (insolação)

> 📌 Sem um heliógrafo na fazenda, estime quantas horas de brilho solar
> (n) esse dia teve, a partir da amplitude térmica (Tmax − Tmin).

In [15]:
insol = amp.insolacao(N, Tmax, Tmin, lat)
print(f"Número de horas de insolação estimado: {insol:.2f} horas")

Número de horas de insolação estimado: 9.84 horas


### 2.12 `Qg_angstrom` — Radiação solar global (Angström-Prescott)

> 📌 Combine a insolação estimada com Qo para obter a radiação solar global
> que efetivamente chega à superfície.

In [16]:
Qg_AP = amp.Qg_angstrom(insol, N, Qo, lat)
print(f"Radiação solar global — Angström-Prescott (Qg_AP): {Qg_AP:.2f} MJ/m² dia")

Radiação solar global — Angström-Prescott (Qg_AP): 22.83 MJ/m² dia


### 2.13 `Qg_hargreaves` — Radiação solar global (Hargreaves-Samani)

> 📌 E se você não tiver nem a insolação estimada? O método de
> Hargreaves-Samani usa só Tmax, Tmin e Qo.

In [17]:
Qg_HS = amp.Qg_hargreaves(Tmax, Tmin, Qo)
print(f"Radiação solar global — Hargreaves-Samani (Qg_HS): {Qg_HS:.2f} MJ/m² dia")

Radiação solar global — Hargreaves-Samani (Qg_HS): 19.23 MJ/m² dia


---
## 3️⃣ Temperatura

### 3.1 `temp_media_extremos` — Temperatura média pelos extremos

> 📌 O método mais simples e mais comum: a média entre Tmax e Tmin do dia.

In [18]:
Tmed = amp.temp_media_extremos(Tmax, Tmin)
print(f"Temperatura média diária (extremos): {Tmed:.2f} °C")

Temperatura média diária (extremos): 20.85 °C


### 3.2 `temp_media_estacao_automatica` — Temperatura média por observações horárias

> 📌 A estação automática da fazenda registrou 6 leituras ao longo do dia.
> Qual a temperatura média real desse período?

In [19]:
temperaturas_horarias = [16.8, 19.4, 23.9, 27.1, 26.3, 20.6]
Tmed_auto = amp.temp_media_estacao_automatica(temperaturas_horarias)
print(f"Temperatura média (estação automática): {Tmed_auto:.2f} °C")

Temperatura média (estação automática): 22.35 °C


---
## 4️⃣ Umidade do Ar

Usando as temperaturas extremas (`Tmax`, `Tmin`) e a umidade relativa
(`UR = 58%`) do nosso cenário, vamos calcular toda a cadeia de variáveis de
umidade.

### 4.1 `es_tetens` — Pressão de saturação de vapor

> 📌 A pressão de saturação varia com a temperatura, então o mais correto é
> calculá-la separadamente para Tmax e Tmin — e usar a **média das duas**
> como `es` do dia, em vez de aplicar Tetens direto sobre a temperatura
> média.

In [20]:
es_tmax = amp.es_tetens(Tmax)
es_tmin = amp.es_tetens(Tmin)
es = (es_tmax + es_tmin) / 2

print(f"Pressão de saturação em Tmax (es_tmax): {es_tmax:.4f} kPa")
print(f"Pressão de saturação em Tmin (es_tmin): {es_tmin:.4f} kPa")
print(f"Pressão de saturação média (es): {es:.4f} kPa")

Pressão de saturação em Tmax (es_tmax): 3.6710 kPa
Pressão de saturação em Tmin (es_tmin): 1.6194 kPa
Pressão de saturação média (es): 2.6452 kPa


### 4.2 `ea_umidade` — Pressão parcial (atual) de vapor

> 📌 Com UR = 58%, qual a pressão de vapor realmente exercida na atmosfera?

In [21]:
ea = amp.ea_umidade(es, UR)
print(f"Pressão parcial de vapor (ea): {ea:.4f} kPa")

Pressão parcial de vapor (ea): 1.5342 kPa


### 4.3 `deficit_saturacao` — Déficit de saturação de vapor

> 📌 O quão "longe" o ar está da saturação? Esse déficit é proporcional ao
> poder evaporante da atmosfera.

In [22]:
delta_e = amp.deficit_saturacao(es, ea)
print(f"Déficit de saturação (delta_e): {delta_e:.4f} kPa")

Déficit de saturação (delta_e): 1.1110 kPa


### 4.4 `patm_altitude` — Pressão atmosférica pela altitude

> 📌 A fazenda está a 875 m de altitude. Qual a pressão atmosférica local?

In [23]:
Patm = amp.patm_altitude(alt)
print(f"Pressão atmosférica (Patm): {Patm:.4f} kPa")

Pressão atmosférica (Patm): 91.3757 kPa


### 4.5 `umidade_absoluta` — Umidade absoluta do ar

> 📌 Quantos gramas de água há em cada m³ de ar, de fato (não no limite de
> saturação)?

In [24]:
UA = amp.umidade_absoluta(ea, Tmed)
print(f"Umidade absoluta (UA): {UA:.2f} g/m³")

Umidade absoluta (UA): 11.31 g/m³


### 4.6 `umidade_saturacao` — Umidade de saturação do ar

> 📌 E quantos gramas de água por m³ o ar poderia reter no máximo, nessa
> mesma temperatura?

In [25]:
US = amp.umidade_saturacao(es, Tmed)
print(f"Umidade de saturação (US): {US:.2f} g/m³")

Umidade de saturação (US): 19.51 g/m³


### 4.7 `umidade_relativa` — Umidade relativa (conferência)

> 📌 Recalculando UR a partir de `ea` e `es` — o resultado deve bater com os
> 58% que usamos como dado de entrada.

In [26]:
UR_calculada = amp.umidade_relativa(ea, es)
print(f"Umidade relativa recalculada: {UR_calculada:.2f} %")

Umidade relativa recalculada: 58.00 %


### 4.8 `ponto_orvalho` — Temperatura do ponto de orvalho

> 📌 A que temperatura o ar precisaria esfriar, mantendo o mesmo teor de
> vapor, para começar a formar orvalho?

In [27]:
To = amp.ponto_orvalho(ea)
print(f"Temperatura do ponto de orvalho (To): {To:.2f} °C")

Temperatura do ponto de orvalho (To): 13.37 °C


### 4.9 `constante_psicrometrica` — Constante psicrométrica (gamma)

> 📌 Essencial nos métodos combinados de evapotranspiração (Priestley-Taylor,
> Penman-Monteith) — depende só da pressão atmosférica local.

In [28]:
gamma = amp.constante_psicrometrica(Patm)
print(f"Constante psicrométrica (gamma): {gamma:.6f} kPa/°C")

Constante psicrométrica (gamma): 0.060765 kPa/°C


---
## 5️⃣ Balanço de Energia

### 5.1 `boc_saldo` — Balanço de ondas curtas

> 📌 Da radiação solar global que chega (`Qg_HS`, calculada na seção 2),
> quanto sobra depois de descontar o albedo da superfície (r = 0,23, típico
> de vegetação)?

In [29]:
BOC = amp.boc_saldo(Qg_HS, r=0.23)
print(f"Balanço de ondas curtas (BOC): {BOC:.2f} MJ/m² dia")

Balanço de ondas curtas (BOC): 14.81 MJ/m² dia


### 5.2 `bol_saldo` — Balanço de ondas longas

> 📌 Quanta energia a superfície perde por emissão de radiação infravermelha,
> corrigida pela nebulosidade e pela umidade do ar?

In [30]:
Qg_cs = (0.75 + 2e-5 * alt) * Qo  # radiação de céu claro (FAO-56)
BOL = amp.bol_saldo(Tmax, Tmin, ea, Qg_HS, Qg_cs)
print(f"Radiação de céu claro (Qg_cs): {Qg_cs:.2f} MJ/m² dia")
print(f"Balanço de ondas longas (BOL): {BOL:.4f} MJ/m² dia")

Radiação de céu claro (Qg_cs): 25.29 MJ/m² dia
Balanço de ondas longas (BOL): -4.1401 MJ/m² dia


### 5.3 `saldo_radiacao` — Saldo de radiação (Rn)

> 📌 Somando os dois balanços (curtas + longas), quanta energia líquida
> sobra na superfície para os processos físicos e biológicos do dia?

In [31]:
Rn = amp.saldo_radiacao(BOC, BOL)
print(f"Saldo de radiação (Rn): {Rn:.2f} MJ/m² dia")

Saldo de radiação (Rn): 10.67 MJ/m² dia


---
## 6️⃣ Evapotranspiração

Primeiro, dois métodos **mensais** (precisam de uma série de 12 temperaturas
médias) — úteis quando você só tem normais climatológicas. Depois, os
métodos **diários**, que reaproveitam `Rn`, `gamma` e as demais variáveis
calculadas até aqui.

### 6.1 `thornthwaite_mensal` — ETP mensal por Thornthwaite (1948)

> 📌 Com as temperaturas médias mensais históricas de Ponta Grossa, estime a
> evapotranspiração potencial de cada mês do ano.

In [32]:
meses = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
T_mensal = [22.1, 22.4, 21.6, 19.0, 15.8, 14.2, 13.9, 15.4, 17.1, 19.3, 20.7, 21.9]

df_mensal = pd.DataFrame({'Mes': meses, 'T_media_C': T_mensal})
df_thornthwaite = amp.thornthwaite_mensal(df_mensal, lat=lat)
df_thornthwaite

,Mes,T_media_C,ETP_mm_mes
0,Jan,22.1,108.90
1,Fev,22.4,96.83
2,Mar,21.6,94.63
3,Abr,19.0,66.96
4,Mai,15.8,45.65
5,Jun,14.2,34.85
6,Jul,13.9,35.01
7,Ago,15.4,44.90
8,Set,17.1,56.72
9,Out,19.3,78.84


### 6.2 `camargo_maluf_mensal` — ETP mensal por Camargo (modif. Maluf)

> 📌 Compare com um segundo método mensal, que usa a radiação extraterrestre
> em vez do índice térmico de Thornthwaite.

In [33]:
df_camargo = amp.camargo_maluf_mensal(df_mensal, lat=lat)
df_camargo

,Mes,T_media_C,Qo_MJ_m2dia,Qo_mm_dia,ETP_mm_mes
0,Jan,22.1,42.56,17.37,118.97
1,Fev,22.4,39.88,16.27,102.05
2,Mar,21.6,35.40,14.45,96.72
3,Abr,19.0,29.37,11.98,68.30
4,Mai,15.8,24.02,9.80,48.01
5,Jun,14.2,21.45,8.75,37.27
6,Jul,13.9,22.49,9.17,39.53
7,Ago,15.4,26.86,10.96,52.31
8,Set,17.1,32.77,13.37,68.59
9,Out,19.3,38.16,15.57,93.15


### 6.3 `declive_pressao_vapor` — Declive da curva de pressão de vapor (Delta)

> 📌 Voltando ao dia 15/09: qual o declive da curva de Tetens na temperatura
> média do dia? É um insumo dos métodos combinados a seguir.

In [34]:
Delta = amp.declive_pressao_vapor(Tmed)
print(f"Declive da curva de pressão de vapor (Delta): {Delta:.6f} kPa/°C")

Declive da curva de pressão de vapor (Delta): 0.151523 kPa/°C


### 6.4 `etp_hargreaves_samani` — ETP por Hargreaves & Samani (1985)

> 📌 Estime a ETP diária usando só a amplitude térmica e a radiação
> extraterrestre — útil quando faltam dados de vento e umidade.

In [35]:
ETP_HS = amp.etp_hargreaves_samani(Qo, Tmax, Tmin, Tmed)
print(f"ETP — Hargreaves-Samani: {ETP_HS:.2f} mm/dia")

ETP — Hargreaves-Samani: 4.36 mm/dia


### 6.5 `etp_priestley_taylor` — ETP por Priestley & Taylor (1972)

> 📌 Agora com o saldo de radiação (`Rn`) e a constante psicrométrica
> (`gamma`) já calculados, aplique o método simplificado de Priestley-Taylor
> (bom para superfícies bem supridas de água).

In [36]:
G = 0  # fluxo de calor no solo, desprezado em escala diária
ETP_PT = amp.etp_priestley_taylor(Rn, G, Delta, gamma)
print(f"ETP — Priestley-Taylor: {ETP_PT:.2f} mm/dia")

ETP — Priestley-Taylor: 3.91 mm/dia


### 6.6 `eto_penman_monteith_fao56` — ETo por Penman-Monteith (padrão FAO-56)

> 📌 O método-padrão internacional: combine radiação, vento (`u2 = 1.8 m/s`)
> e déficit de vapor para obter a evapotranspiração de referência.

In [37]:
ETo_PM = amp.eto_penman_monteith_fao56(Rn, G, Tmed, u2, es, ea, Delta, gamma)
print(f"ETo — Penman-Monteith FAO-56: {ETo_PM:.2f} mm/dia")

ETo — Penman-Monteith FAO-56: 4.13 mm/dia


---
## 7️⃣ Grau-Dias

Mudando de cenário: agora somos o agrônomo responsável por um talhão de
**soja** (Tb = 14 °C, constante térmica do ciclo CT = 1350 °C·dia), com
temperaturas médias mensais previstas para o ano.

### 7.1 `data_maturacao_fisiologica` — Quando a soja vai maturar?

> 📌 A soja foi semeada em 5 de outubro. Por acúmulo de graus-dia mês a mês,
> em que data ela deve atingir a maturação fisiológica?

In [43]:
meses_num = list(range(1, 13))
Tmed_mensal = [23.1, 23.3, 22.4, 20.5, 17.2, 15.6, 15.3, 16.9, 18.6, 20.5, 21.7, 22.8]
Tmin_mensal = [18.9, 19.0, 17.8, 15.7, 12.3, 10.8, 10.2, 11.6, 13.4, 15.6, 16.9, 18.0]
Tmax_mensal = [27.6, 27.9, 27.3, 25.6, 22.4, 20.7, 20.7, 22.5, 24.1, 25.7, 26.8, 27.4]

Tb, CT = 14, 1350
dia_semeadura, mes_semeadura = 15, 10  # 5 de outubro

df_clima = pd.DataFrame({
    'dia': [1] * 12,
    'mes': meses_num,
    'Tmed': Tmed_mensal,
    'Tmax': Tmax_mensal,
    'Tmin': Tmin_mensal,
})

resultado_maturacao = amp.data_maturacao_fisiologica(
    df_clima, Tb, CT, dia_semeadura, mes_semeadura, intervalo='M'
)
resultado_maturacao

Data de semeadura: 15 de outubro
Data de maturação fisiológica: 24 de março


,data,GD_ciclo
0,2023-10-15,104.0
1,2023-11-01,335.0
2,2023-12-01,607.8
3,2023-01-01,889.9
4,2023-02-01,1150.3
5,2023-03-01,1410.7


### 7.2 `data_semeadura` — Quando semear para colher numa data-alvo?

> 📌 Agora o problema inverso: você quer colher por volta de 20 de março do
> próximo ano. Em que data a soja precisa ser semeada?

In [39]:
dia_maturacao, mes_maturacao = 20, 3  # colheita-alvo: 20 de março

resultado_semeadura = amp.data_semeadura(
    df_clima, Tb=10, CT=1350,
    dia_maturacao=dia_maturacao, mes_maturacao=mes_maturacao,
    intervalo='M',
)
resultado_semeadura

Data de maturação (referência): 20 de março
Data de semeadura necessária: 26 de dezembro


,data,GD_ciclo
0,2023-03-01,248.0
1,2023-02-01,620.4
2,2023-01-01,1026.5
3,2023-12-01,1423.3


---
## 8️⃣ Balanço Hídrico

### 8.1 `balanco_hidrico_climatologico` — Balanço hídrico climatológico anual

> 📌 Usando a chuva média mensal da região e a ETP mensal calculada na
> seção 6 (Thornthwaite), monte o balanço hídrico climatológico da fazenda
> (Thornthwaite & Mather), com capacidade de água disponível no solo de
> 125 mm.

In [40]:
precipitacao_mensal = [165.2, 148.7, 132.4, 88.6, 78.3, 118.5,
                       132.9, 95.4, 138.7, 158.2, 142.6, 172.3]

df_bhc_entrada = pd.DataFrame({
    'Meses': meses,
    'P (mm/mês)': precipitacao_mensal,
    'ETP (mm/mês)': df_thornthwaite['ETP_mm_mes'],
})

df_balanco_climatologico = amp.balanco_hidrico_climatologico(df_bhc_entrada, CAD=125.0)
df_balanco_climatologico[['Meses', 'P (mm/mês)', 'ETP (mm/mês)', 'ARM (mm/mês)',
                           'ETR (mm/mês)', 'DEF (mm/mês)', 'EXC (mm/mês)']]

,Meses,P (mm/mês),ETP (mm/mês),ARM (mm/mês),ETR (mm/mês),DEF (mm/mês),EXC (mm/mês)
0,Jan,165.2,108.90,125.0,108.90,0.0,56.30
1,Fev,148.7,96.83,125.0,96.83,0.0,51.87
2,Mar,132.4,94.63,125.0,94.63,0.0,37.77
3,Abr,88.6,66.96,125.0,66.96,0.0,21.64
4,Mai,78.3,45.65,125.0,45.65,0.0,32.65
5,Jun,118.5,34.85,125.0,34.85,0.0,83.65
6,Jul,132.9,35.01,125.0,35.01,0.0,97.89
7,Ago,95.4,44.90,125.0,44.90,0.0,50.50
8,Set,138.7,56.72,125.0,56.72,0.0,81.98
9,Out,158.2,78.84,125.0,78.84,0.0,79.36


### 8.2 `balanco_hidrico_cultura` — Balanço hídrico de cultura

> 📌 Zoom no ciclo da soja: com a chuva, a evapotranspiração da cultura
> (ETc = Kc × ETo) e a capacidade de água disponível crescendo junto com as
> raízes, mês a mês, calcule o balanço hídrico específico da lavoura — e o
> Índice de Satisfação das Necessidades de Água (ISNA).

In [41]:
df_bhc_cultura_entrada = pd.DataFrame({
    'Chuva': [52.3, 41.8, 28.6, 18.2, 65.4, 88.1],
    'ETc':   [12.5, 22.8, 38.4, 51.2, 47.6, 30.1],
    'CAD':   [18.0, 30.0, 42.0, 54.0, 54.0, 54.0],
})

df_balanco_cultura = amp.balanco_hidrico_cultura(df_bhc_cultura_entrada)
df_balanco_cultura

,Chuva,ETc,CAD,P-ETc,ARM,ALT,ETR,DEF,EXC,ISNA
0,52.3,12.5,18.0,39.8,18.000000,0.000000,12.500000,0.000000,39.800000,1.000000
1,41.8,22.8,30.0,19.0,30.000000,12.000000,22.800000,0.000000,7.000000,1.000000
2,28.6,38.4,42.0,-9.8,23.756687,-6.243313,34.843313,3.556687,0.000000,0.907378
3,18.2,51.2,54.0,-33.0,12.893882,-10.862805,29.062805,22.137195,0.000000,0.567633
4,65.4,47.6,54.0,17.8,30.693882,17.800000,47.600000,0.000000,0.000000,1.000000
5,88.1,30.1,54.0,58.0,54.000000,23.306118,30.100000,0.000000,34.693882,1.000000


---
## 🎉 Fim do tour

Você acabou de rodar **todas as 41 funções públicas** do `agrometeorologiapy`,
do começo ao fim, com um único cenário coerente.

- 📦 PyPI: <https://pypi.org/project/agrometeorologiapy/>
- 💻 Código-fonte: <https://github.com/fcoliveira-utfpr/agrometeorologiapy>
- 🐛 Encontrou um problema? Abra uma *issue*: <https://github.com/fcoliveira-utfpr/agrometeorologiapy/issues>

```bash
pip install agrometeorologiapy
```

Licença BSD-3-Clause.